# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srinivas25046/FlyRank-MLstarter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Lane:** Refresh / Content Opportunity Scoring. **Question:** given a large portfolio of
content pages, which ones are worth reviewing for a refresh first, and how much better than
guesswork can a data-driven priority queue actually get? **Decision supported:** a content
team's monthly review cycle -- what to look at first, not a fully automated action. **Unit of
analysis:** one (client, content) pair per month. **Why ML/data helps:** the real decision is an
ordering across many correlated, non-linearly-interacting signals (traffic size, age, position,
within-month trend) -- exactly the "too many combinations to hand-tune" case where a validated
model can outperform a fixed rule, which w04's baseline already showed clearing only a thin
1.15x lift over guessing.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("See markdown above. Full reasoning: work/notebooks/w01_research_question.ipynb and w02_ml_task_framing.ipynb")

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

FlyRank ML Internship warehouse (Hugging Face, gated). Daily performance fact table
(partitioned by month) joined to client and content dimension tables. Feature month: February
2026. Label month: March 2026 (a genuinely forward-looking outcome, not a same-window relabel).
**Excluded:** the sealed final month (June 2026, and its `_sample` companion) -- reserved as a
held-out check, never used for feature/label development; rows before a client's own tracking
start date, excluded via availability flags rather than treated as zero activity. **Public-safe:**
every identifier used anywhere in this notebook and the deployed paper is a pseudonymous hash
from FlyRank's own release -- no client name, domain, or real query appears anywhere.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb scikit-learn matplotlib

import duckdb
import json
import numpy as np
import pandas as pd
from getpass import getpass
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = getpass("Hugging Face READ token (from a Colab Secret named HF_TOKEN ideally): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT = f"{BASE}/dim_content.parquet"
DAILY_FACT = f"{BASE}/fact_content_daily_performance/**/*.parquet"
FEATURE_MONTH = "2026-02"
LABEL_MONTH = "2026-03"

feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS feb_impressions,
           SUM(gsc_clicks)      AS feb_clicks,
           AVG(gsc_avg_position) AS feb_avg_position,
           SUM(ga4_sessions)     AS feb_sessions,
           SUM(sessions_ai)      AS feb_ai_sessions,
           SUM(CASE WHEN report_date < DATE '{FEATURE_MONTH}-15' THEN gsc_impressions ELSE 0 END) AS feb_h1,
           SUM(CASE WHEN report_date >= DATE '{FEATURE_MONTH}-15' THEN gsc_impressions ELSE 0 END) AS feb_h2
    FROM read_parquet('{DAILY_FACT}', hive_partitioning=true)
    WHERE month = '{FEATURE_MONTH}'
    GROUP BY client_hash_id, content_hash_id
""").df()

label = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS mar_impressions
    FROM read_parquet('{DAILY_FACT}', hive_partitioning=true)
    WHERE month = '{LABEL_MONTH}'
    GROUP BY client_hash_id, content_hash_id
""").df()

panel = feat.merge(label, on=["client_hash_id", "content_hash_id"], how="inner")
panel = panel[panel["feb_impressions"] > 0].copy()
panel["declined_next_month"] = (panel["mar_impressions"] < panel["feb_impressions"] * 0.8).astype(int)

content_meta = con.sql(f"SELECT content_hash_id, content_created_date FROM read_parquet('{DIM_CONTENT}')").df()
panel = panel.merge(content_meta, on="content_hash_id", how="left")
panel["content_created_date"] = pd.to_datetime(panel["content_created_date"])
as_of = pd.Timestamp(f"{FEATURE_MONTH}-28")
panel["age_days"] = (as_of - panel["content_created_date"]).dt.days
panel["declining_now"] = (panel["feb_h2"] < panel["feb_h1"] * 0.8).astype(int)

print(f"Panel: {len(panel):,} (client, content) pairs, {panel['client_hash_id'].nunique()} clients")
print(f"Base rate (declined_next_month): {panel['declined_next_month'].mean():.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Panel: 145,279 (client, content) pairs, 42 clients
Base rate (declined_next_month): 26.1%


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Task type:** ranking/scoring, not classification -- the decision is "which ones first."
**Label:** `declined_next_month`, a defined proxy (March impressions down 20%+ vs. February),
not an observed "needed a refresh" judgment. **Features** (all February-only, never touching
March): `feb_impressions`, `feb_clicks`, `feb_avg_position`, `feb_sessions`, `feb_ai_sessions`,
`age_days`, `declining_now`. **Baseline:** a transparent rule -- flag pages in an empirically
checked 90-365 day age band with >=500 monthly impressions, score by raw impression volume.
**Validation:** every model result reported under both a naive random row split and a
client-held-out (`GroupShuffleSplit`) split, specifically to catch client-identity memorization.
**Leakage checks:** a static column-set check plus a dynamic injection test (add the literal
March outcome as a "feature," confirm precision spikes to ~100%, confirm it collapses back down
once removed).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

FEATURE_COLS = ["feb_impressions", "feb_clicks", "feb_avg_position", "feb_sessions", "feb_ai_sessions", "age_days", "declining_now"]
LABEL_COL = "declined_next_month"

panel["stale"] = ((panel["age_days"] >= 90) & (panel["age_days"] < 365)).astype(int)
panel["visible"] = (panel["feb_impressions"] >= 500).astype(int)
panel["baseline_score"] = panel["stale"] * panel["visible"] * panel["feb_impressions"]

model_df = panel.dropna(subset=FEATURE_COLS + [LABEL_COL]).copy()
print(f"Rows usable for modeling (complete features): {len(model_df):,} / {len(panel):,}")
missing_by_col = panel[FEATURE_COLS].isna().sum()
print("\nMissing-value counts by feature:")
print(missing_by_col.to_string())

Rows usable for modeling (complete features): 72,849 / 145,279

Missing-value counts by feature:
feb_impressions         0
feb_clicks              0
feb_avg_position        0
feb_sessions        72430
feb_ai_sessions     72430
age_days                0
declining_now           0


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()


def fit_and_score(train_df, test_df):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[FEATURE_COLS])
    X_test = scaler.transform(test_df[FEATURE_COLS])
    y_train, y_test = train_df[LABEL_COL].values, test_df[LABEL_COL].values

    logreg = LogisticRegression(random_state=RANDOM_SEED, max_iter=1000).fit(X_train, y_train)
    logreg_scores = logreg.predict_proba(X_test)[:, 1]

    rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_SEED, n_jobs=-1).fit(X_train, y_train)
    rf_scores = rf.predict_proba(X_test)[:, 1]

    baseline_scores = test_df["baseline_score"].values
    base_rate = y_test.mean()
    rows = []
    for k in (20, 50):
        rows.append({
            "k": k, "base_rate": base_rate,
            "baseline_precision": precision_at_k(baseline_scores, y_test, k),
            "logreg_precision": precision_at_k(logreg_scores, y_test, k),
            "rf_precision": precision_at_k(rf_scores, y_test, k),
        })
    return pd.DataFrame(rows)


# Grouped split (honest)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
tr_idx, te_idx = next(gss.split(model_df, groups=model_df["client_hash_id"]))
train_g, test_g = model_df.iloc[tr_idx], model_df.iloc[te_idx]
results_grouped = fit_and_score(train_g, test_g)
results_grouped["split"] = "grouped_by_client"

# Random split (naive, for comparison)
train_r, test_r = train_test_split(model_df, test_size=0.2, random_state=RANDOM_SEED)
results_random = fit_and_score(train_r, test_r)
results_random["split"] = "random_row_split"

comparison = pd.concat([results_grouped, results_random], ignore_index=True)
display_table = comparison.copy()
for col in ["base_rate", "baseline_precision", "logreg_precision", "rf_precision"]:
    display_table[col] = display_table[col].map(lambda x: f"{x:.1%}")
print(display_table[["split", "k", "base_rate", "baseline_precision", "logreg_precision", "rf_precision"]].to_string(index=False))

            split  k base_rate baseline_precision logreg_precision rf_precision
grouped_by_client 20     40.0%              20.0%            85.0%        70.0%
grouped_by_client 50     40.0%              20.0%            78.0%        70.0%
 random_row_split 20     30.5%              25.0%            40.0%       100.0%
 random_row_split 50     30.5%              20.0%            50.0%        98.0%


## 5. Limitations

*What this work cannot claim.*

Cross-sectional, single-cohort data (one Feb->March transition) -- no experiment was run, so
nothing here claims refreshing *causes* recovery. The logistic-regression grouped-split number
rests on only 5 held-out clients and is reported as promising, not proven; the Random Forest's
70% grouped-split number is the more robust floor. Roughly half the panel has no usable GA4
signal and is excluded from scoring, not guessed at. No February-only feature set can foresee a
decline with no February-internal precursor. Every identifier is a pseudonymous hash -- nothing
client-identifying appears anywhere in this repo.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

n_test_clients_grouped = test_g["client_hash_id"].nunique()
n_overlap_random = len(set(train_r["client_hash_id"]) & set(test_r["client_hash_id"]))
print(f"Grouped-split test clients (the samplce size behind the logreg ceiling number): {n_test_clients_grouped}")
print(f"Random-split overlapping clients (the memorization risk made concrete): {n_overlap_random}")
print(f"Unscorable rows (missing GA4 features, excluded from scoring): {len(panel) - len(model_df):,} "
      f"({(len(panel) - len(model_df)) / len(panel):.1%} of the full panel)")

Grouped-split test clients (the samplce size behind the logreg ceiling number): 5
Random-split overlapping clients (the memorization risk made concrete): 23
Unscorable rows (missing GA4 features, excluded from scoring): 72,430 (49.9% of the full panel)


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Final production scorer: logistic regression retrained on the full scorable panel (not held
# out -- the split above was for honest evaluation, not for withholding data from deployment).
scorable = model_df.copy()
scaler_final = StandardScaler()
X_all = scaler_final.fit_transform(scorable[FEATURE_COLS])
final_model = LogisticRegression(random_state=RANDOM_SEED, max_iter=1000).fit(X_all, scorable[LABEL_COL].values)
scorable["model_score"] = final_model.predict_proba(X_all)[:, 1]


def reason_code(row):
    if row["declining_now"] and row["visible"]:
        return "actively_declining_visible"
    if row["stale"] and row["visible"]:
        return "stale_visible_page"
    if not row["visible"]:
        return "low_volume_uncertain"
    return "other"


scorable["reason_code"] = scorable.apply(reason_code, axis=1)

VOLUME_FLOOR = 500
scorable["meets_volume_floor"] = scorable["feb_impressions"] >= VOLUME_FLOOR
rankable = scorable[scorable["meets_volume_floor"]].copy()
below_floor = scorable[~scorable["meets_volume_floor"]].copy()

deciles = pd.qcut(rankable["model_score"], 10, labels=False, duplicates="drop")
n_tiers = deciles.max() + 1
rankable["action_tier"] = np.select(
    [deciles >= n_tiers - 1, deciles >= n_tiers - 3], ["refresh_priority", "review"], default="monitor"
)
below_floor["action_tier"] = "insufficient_volume"
scorable = pd.concat([rankable, below_floor], ignore_index=True)

tier_counts = scorable["action_tier"].value_counts().to_dict()
print("Action tier counts:")
print(scorable["action_tier"].value_counts().to_string())

no_evidence = rankable[(rankable["action_tier"] == "refresh_priority") & (rankable["declining_now"] == 0)]
print(f"\n'refresh_priority' rows lacking internal decline evidence (should be low): "
      f"{len(no_evidence):,} / {(rankable['action_tier']=='refresh_priority').sum():,}")

Action tier counts:
action_tier
insufficient_volume    47908
monitor                17459
review                  4988
refresh_priority        2494

'refresh_priority' rows lacking internal decline evidence (should be low): 0 / 2,494


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --- The exact figure the deployed paper's Results section describes ---
fig_data = comparison.pivot(index="split", columns="k", values=["logreg_precision", "rf_precision"])
plot_rows = comparison[comparison["k"] == 20].set_index("split")

fig, ax = plt.subplots(figsize=(7, 4.5))
splits_order = ["random_row_split", "grouped_by_client"]
x = np.arange(len(splits_order))
width = 0.35
logreg_vals = [plot_rows.loc[s, "logreg_precision"] for s in splits_order]
rf_vals = [plot_rows.loc[s, "rf_precision"] for s in splits_order]
ax.bar(x - width / 2, logreg_vals, width, label="Logistic Regression", color="#4d6142")
ax.bar(x + width / 2, rf_vals, width, label="Random Forest", color="#9a5027")
ax.axhline(0.20, color="gray", linestyle="--", linewidth=1, label="Baseline (w04)")
ax.set_xticks(x)
ax.set_xticklabels(["Random\n(naive)", "Grouped by\nclient (honest)"])
ax.set_ylabel("Precision@20")
ax.set_title("Naive vs. honest split precision -- the gap is the finding")
ax.legend()
fig.tight_layout()

fig_path = Path("work/figures/precision_comparison.png")
fig_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_path, dpi=150)
plt.close(fig)
print(f"Wrote {fig_path}")

# --- The consolidated metrics JSON the whole paper traces back to ---
capstone_metrics = {
    "feature_month": FEATURE_MONTH,
    "label_month": LABEL_MONTH,
    "random_seed": RANDOM_SEED,
    "panel_n": int(len(panel)),
    "panel_n_clients": int(panel["client_hash_id"].nunique()),
    "panel_base_rate": float(panel["declined_next_month"].mean()),
    "model_df_n": int(len(model_df)),
    "n_unscorable": int(len(panel) - len(model_df)),
    "comparison_table": comparison.to_dict(orient="records"),
    "grouped_split_test_clients": int(n_test_clients_grouped),
    "random_split_overlapping_clients": int(n_overlap_random),
    "final_deployed_model": "logistic_regression_full_panel_retrain",
    "volume_floor_impressions": VOLUME_FLOOR,
    "action_tier_counts": {k: int(v) for k, v in tier_counts.items()},
}
metrics_path = Path("work/outputs/capstone_metrics.json")
metrics_path.parent.mkdir(parents=True, exist_ok=True)
with open(metrics_path, "w") as f:
    json.dump(capstone_metrics, f, indent=2, default=str)
print(f"Wrote {metrics_path} -- commit this one, it's the receipt the deployed paper's numbers trace back to.")

Wrote work/figures/precision_comparison.png
Wrote work/outputs/capstone_metrics.json -- commit this one, it's the receipt the deployed paper's numbers trace back to.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.